# Feature Engineering: Lags, Rolling Windows & Hydrology


In [1]:
import os
import numpy as np
import pandas as pd

# read preprocessed dataset
data_path = os.path.join('..', 'data', 'processed', 'flood_data_preprocessed.csv')
df = pd.read_csv(data_path)
print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
df.head(3)


Loaded 20544 rows, 20 columns



  Station_Names  Year  Month  ...  Rainfall log  Wind_Speed log   ALT log
0       Barisal  1949      1  ...      0.000000        0.374115  1.609438
1       Barisal  1949      2  ...      2.302585        0.506371  1.609438
2       Barisal  1949      3  ...      2.197225        0.734858  1.609438

[3 rows x 20 columns]


### 1. Antecedent rainfall lags & rolling saturation


In [2]:
# make sure data is ordered chronologically per station
df = df.sort_values(by=['Station_Names', 'Year', 'Month']).reset_index(drop=True)

# 1-month and 2-month lag of rainfall
df['Rainfall_Lag1'] = df.groupby('Station_Names')['Rainfall'].shift(1).fillna(0)
df['Rainfall_Lag2'] = df.groupby('Station_Names')['Rainfall'].shift(2).fillna(0)

# 3-month rolling rainfall sum for cumulative saturation
df['Rainfall_Rolling3M'] = df.groupby('Station_Names')['Rainfall'].transform(
    lambda s: s.rolling(3, min_periods=1).sum()
)

# log transform the new rainfall metrics as well
df['Rainfall_Lag1_log'] = np.log1p(df['Rainfall_Lag1'])
df['Rainfall_Lag2_log'] = np.log1p(df['Rainfall_Lag2'])
df['Rainfall_Rolling3M_log'] = np.log1p(df['Rainfall_Rolling3M'])

print("Added rainfall lag and rolling features.")
df[['Station_Names', 'Year', 'Month', 'Rainfall', 'Rainfall_Lag1', 'Rainfall_Rolling3M']].head(5)


Added rainfall lag and rolling features.



  Station_Names  Year  Month  Rainfall  Rainfall_Lag1  Rainfall_Rolling3M
0       Barisal  1949      1       0.0            0.0                 0.0
1       Barisal  1949      2       9.0            0.0                 9.0
2       Barisal  1949      3       8.0            9.0                17.0
3       Barisal  1949      4     140.0            8.0               157.0
4       Barisal  1949      5     217.0          140.0               365.0


### 2. Runoff risk & moisture interaction


In [3]:
# low elevation with high rain pools faster
df['Runoff_Ratio'] = df['Rainfall'] / (df['ALT'] + 1.0)
df['Runoff_Ratio_log'] = np.log1p(df['Runoff_Ratio'])

# moisture saturation proxy from humidity and clouds
df['Moisture_Index'] = (df['Relative_Humidity'] * df['Cloud_Coverage']) / 100.0

print("Added runoff ratio and moisture index.")
df[['Rainfall', 'ALT', 'Runoff_Ratio', 'Relative_Humidity', 'Cloud_Coverage', 'Moisture_Index']].head(3)


Added runoff ratio and moisture index.



   Rainfall  ALT  ...  Cloud_Coverage  Moisture_Index
0       0.0    4  ...             0.6           0.408
1       9.0    4  ...             0.9           0.567
2       8.0    4  ...             1.5           0.885

[3 rows x 6 columns]


### 3. Cyclical seasonality & monsoon flag


In [4]:
# cyclical encoding so month 12 wraps to month 1
df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)

# monsoon season flag (June to September)
df['Is_Monsoon'] = df['Month'].isin([6, 7, 8, 9]).astype(int)

print("Added seasonal indicators.")
print("Monsoon records:", (df['Is_Monsoon'] == 1).sum(), "out of", len(df))


Added seasonal indicators.
Monsoon records: 6848 out of 20544



### 4. Station encoding & save dataset


In [5]:
# one-hot encode station names for tree models
station_dummies = pd.get_dummies(df['Station_Names'], prefix='Station', drop_first=True, dtype=int)
df_featured = pd.concat([df, station_dummies], axis=1)

print("Final dataset shape:", df_featured.shape)

out_path = os.path.join('..', 'data', 'processed', 'flood_data_featured.csv')
df_featured.to_csv(out_path, index=False)
print(f"Saved to: {out_path} ({os.path.getsize(out_path)/(1024*1024):.2f} MB)")


Final dataset shape: (20544, 64)
Saved to: ..\data\processed\flood_data_featured.csv (6.94 MB)

